<a href="https://colab.research.google.com/github/hrichamaharjan/-Hospital-Performance-Analysis/blob/main/Data_cleaning_EDA_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv('/content/sample_data/jobs_20260506_051611.csv')
df_clean = df.copy()

In [ ]:
import os
os.listdir('/content')

['.config', 'sample_data']

In [ ]:
print(df_clean.shape)
print(df_clean.columns.tolist())
print(df_clean.dtypes)
print(df_clean.isna().sum().sort_values(ascending=False))

(1195, 21)
['Job ID', 'Title', 'Company', 'Industry', 'Employment Type', 'Date Posted', 'Valid Through', 'Location', 'City', 'Region', 'Country', 'Description', 'Salary Currency', 'Salary Min', 'Salary Max', 'Salary Unit', 'Education', 'Experience Months', 'Internal Identifier', 'Canonical URL', 'Parse Status']
Job ID                   int64
Title                   object
Company                 object
Industry                object
Employment Type         object
Date Posted             object
Valid Through           object
Location                object
City                    object
Region                  object
Country                 object
Description             object
Salary Currency         object
Salary Min             float64
Salary Max             float64
Salary Unit             object
Education               object
Experience Months      float64
Internal Identifier     object
Canonical URL           object
Parse Status            object
dtype: object
Parse Status          

In [ ]:
## formatting the column names
df_clean.columns=(df_clean.columns.str.strip().str.lower().str.replace(" ", "_")
                 )
print(df_clean.columns.tolist())


['job_id', 'title', 'company', 'industry', 'employment_type', 'date_posted', 'valid_through', 'location', 'city', 'region', 'country', 'description', 'salary_currency', 'salary_min', 'salary_max', 'salary_unit', 'education', 'experience_months', 'internal_identifier', 'canonical_url', 'parse_status']


In [ ]:
##CHECK duplicate
df_clean["job_id"].duplicated().sum()
dupes = df_clean[df_clean["job_id"].duplicated(keep=False)]
dupes.sort_values("job_id").head(20)

,job_id,title,company,industry,employment_type,date_posted,valid_through,location,city,region,...,description,salary_currency,salary_min,salary_max,salary_unit,education,experience_months,internal_identifier,canonical_url,parse_status


In [ ]:
##converting the date fields from text to datetime
df_clean['date_posted']=pd.to_datetime(df_clean["date_posted"], errors="coerce")
df_clean["valid_through"] = pd.to_datetime(df_clean["valid_through"], errors="coerce")

print(df_clean[["date_posted", "valid_through"]].dtypes)

date_posted      datetime64[ns, UTC]
valid_through    datetime64[ns, UTC]
dtype: object


In [ ]:
## we are making sure the numeric columns are properly formatted
numeric_cols = ["salary_min", "salary_max", "experience_months"]

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print(df_clean[numeric_cols].dtypes)

salary_min           float64
salary_max           float64
experience_months    float64
dtype: object


In [ ]:
##now we clean text columns
text_cols = [
    "title", "company", "industry", "employment_type", "location",
    "city", "region", "country", "education",
    "salary_currency", "salary_unit", "parse_status"
]

for col in text_cols:
    df_clean[col] = df_clean[col].astype("string").str.strip()


In [ ]:
df_clean["job_id"] = df_clean["job_id"].astype(str)

In [ ]:
## Standardize categorical values
employment_map = {
    "FULL_TIME": "Full-time",
    "Full-time": "Full-time",
    "CONTRACTOR": "Contract",
    "Contract": "Contract",
    "PART_TIME": "Part-time",
    "Part-time": "Part-time",
    "VOLUNTEER": "Volunteer",
    "Volunteer": "Volunteer",
    "INTERN": "Intern"
}

df_clean["employment_type"] = df_clean["employment_type"].replace(employment_map)

print(df_clean["employment_type"].value_counts())

employment_type
Full-time     1022
Contract        98
Part-time       38
Intern          14
Volunteer       12
TEMPORARY        8
OTHER            2
Internship       1
Name: count, dtype: Int64


In [ ]:
salary_unit_map = {
    "YEAR": "Year",
    "HOUR": "Hour"
}

df_clean["salary_unit"] = df_clean["salary_unit"].replace(salary_unit_map)

print(df_clean["salary_unit"].value_counts())

salary_unit
Year    293
Hour     52
DAY       1
Name: count, dtype: Int64


In [ ]:
country_map = {
    "US": "United States"
}

df_clean["country"] = df_clean["country"].replace(country_map)

print(df_clean["country"].value_counts())

country
United States    994
Name: count, dtype: Int64


In [ ]:
df_clean["salary_missing_flag"] = df_clean[
    ["salary_min", "salary_max", "salary_unit", "salary_currency"]
].isna().any(axis=1)

print(df_clean["salary_missing_flag"].value_counts())
df_clean["education_missing_flag"] = df_clean["education"].isna()
df_clean["experience_missing_flag"] = df_clean["experience_months"].isna()

print(df_clean[["education_missing_flag", "experience_missing_flag"]].sum())
df_clean["incomplete_scrape_flag"] = df_clean[
    ["city", "region", "country", "date_posted", "valid_through"]
].isna().all(axis=1)

print(df_clean["incomplete_scrape_flag"].value_counts())

salary_missing_flag
True     849
False    346
Name: count, dtype: int64
education_missing_flag     211
experience_missing_flag    517
dtype: int64
incomplete_scrape_flag
False    994
True     201
Name: count, dtype: int64


In [ ]:
##checking invalid salary
df_clean["invalid_salary_order_flag"] = (
    df_clean["salary_min"].notna() &
    df_clean["salary_max"].notna() &
    (df_clean["salary_min"] > df_clean["salary_max"])
)

print(df_clean["invalid_salary_order_flag"].value_counts())
##checking negative salary values
df_clean["negative_salary_flag"] = (
    (df_clean["salary_min"] < 0) | (df_clean["salary_max"] < 0)
)

df_clean["negative_experience_flag"] = df_clean["experience_months"] < 0

print(df_clean[["negative_salary_flag", "negative_experience_flag"]].sum())
##check for  duplicates
df_clean["possible_duplicate_flag"] = df_clean.duplicated(
    subset=["title", "company", "location"],
    keep=False
)

print(df_clean["possible_duplicate_flag"].value_counts())
##** “Potential duplicate job postings were identified using title, company, and location, and flagged rather than removed.”

invalid_salary_order_flag
False    1195
Name: count, dtype: int64
negative_salary_flag        0
negative_experience_flag    0
dtype: int64
possible_duplicate_flag
False    1086
True      109
Name: count, dtype: int64


In [ ]:
def flag_outliers_iqr(series):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

df_clean["salary_min_outlier_flag"] = flag_outliers_iqr(df_clean["salary_min"])
df_clean["salary_max_outlier_flag"] = flag_outliers_iqr(df_clean["salary_max"])
df_clean["experience_outlier_flag"] = flag_outliers_iqr(df_clean["experience_months"])

print(df_clean[[
    "salary_min_outlier_flag",
    "salary_max_outlier_flag",
    "experience_outlier_flag"
]].sum())
## Outliers were identified using the IQR method and retained, as they likely represent valid high-value observations.

salary_min_outlier_flag    13
salary_max_outlier_flag    14
experience_outlier_flag    29
dtype: int64


In [ ]:
df_clean.to_csv("project_final_cleaned_updated.csv", index=False)

## Data Cleaning — Notes

1. Standardized column names (lowercase, underscores)
2. Converted date fields to datetime format
3. Converted numeric fields (salary, experience) to proper types
4. Treated job_id as string (identifier)
5. Removed extra spaces from text fields
6. Standardized categorical values (employment type, country, salary unit)
7. Identified missing values in salary, education, and experience
8. Created indicator flags for missing data
9. Flagged incomplete scrape records (missing location and dates)
10. Checked for invalid records (salary min > max, negative values)
11. No major inconsistencies found in salary or experience
12. Identified potential duplicate job postings (title + company + location)
13. Detected outliers using IQR method
14. Retained outliers as valid extreme observations

In [ ]:
# =============================
# FEATURE ENGINEERING
# =============================

# 1. Salary midpoint
df_clean["salary_midpoint"] = (
    df_clean["salary_min"] + df_clean["salary_max"]
) / 2

# 2. Experience in years
df_clean["experience_years"] = df_clean["experience_months"] / 12

# 3. Python skill indicator
df_clean["python_skill"] = df_clean["description"].str.contains(
    "python", case=False, na=False
)

# 4. SQL skill indicator
df_clean["sql_skill"] = df_clean["description"].str.contains(
    "sql", case=False, na=False
)

# 5. Remote job indicator using regex
df_clean["remote_flag"] = df_clean["description"].str.contains(
    r"\b(remote|work\s?from\s?home|wfh)\b",
    case=False,
    na=False,
    regex=True
)

# Optional: hybrid indicator
df_clean["hybrid_flag"] = df_clean["description"].str.contains(
    r"\bhybrid\b",
    case=False,
    na=False,
    regex=True
)

# Check feature engineering output
print(df_clean[[
    "salary_midpoint",
    "experience_years",
    "python_skill",
    "sql_skill",
    "remote_flag",
    "hybrid_flag"
]].head(10))


/tmp/ipykernel_6628/1946605783.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_clean["remote_flag"] = df_clean["description"].str.contains(


   salary_midpoint  experience_years  python_skill  sql_skill  remote_flag  \
0              NaN               5.0          True       True        False   
1              NaN               3.0          True       True        False   
2              NaN               NaN         False       True         True   
3              NaN               NaN          True      False        False   
4              NaN               NaN          True       True        False   
5              NaN               2.0          True       True        False   
6              NaN               2.0         False       True        False   
7              NaN               2.0         False       True        False   
8              NaN               2.0          True       True         True   
9              NaN               2.0          True       True         True   

   hybrid_flag  
0        False  
1         True  
2         True  
3        False  
4        False  
5        False  
6        False  
7    

## Feature Engineering — Notes

- Created salary_midpoint from salary_min and salary_max
- Converted experience_months to experience_years
- Extracted Python skill indicator from job description
- Extracted SQL skill indicator from job description
- Identified remote jobs using regex patterns (remote, WFH, work from home)
- Optional: separated hybrid roles using keyword detection
- Transformed unstructured text into structured features


In [ ]:


# =============================
# EDA
# =============================

# 1. Salary summary
print("\nSalary Summary:")
print(df_clean["salary_midpoint"].describe())

# 2. Skill counts
print("\nSkill Counts:")
print("Python jobs:", df_clean["python_skill"].sum())
print("SQL jobs:", df_clean["sql_skill"].sum())

# 3. Remote and hybrid counts
print("\nRemote / Hybrid Counts:")
print("Remote jobs:", df_clean["remote_flag"].sum())
print("Hybrid jobs:", df_clean["hybrid_flag"].sum())

# 4. Experience vs salary correlation
print("\nExperience vs Salary Correlation:")
print(df_clean[["experience_years", "salary_midpoint"]].corr())

# 5. Employment type distribution
print("\nEmployment Type Distribution:")
print(df_clean["employment_type"].value_counts())

# 6. Top companies
print("\nTop Companies:")
print(df_clean["company"].value_counts().head(10))

# 7. Top industries
print("\nTop Industries:")
print(df_clean["industry"].value_counts().head(10))

# 8. Average salary by employment type
print("\nAverage Salary by Employment Type:")
print(df_clean.groupby("employment_type")["salary_midpoint"].mean())

# 9. Average salary by industry
print("\nTop Industries by Average Salary:")
print(
    df_clean.groupby("industry")["salary_midpoint"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)


# =============================
# SAVE FINAL FILE
# =============================

df_clean.to_csv("projectfinal_cleaned_featured.csv", index=False)
print("\nFile saved as projectfinal_cleaned_featured.csv")


Salary Summary:
count       346.000000
mean     128264.520318
std      106407.668902
min           0.000000
25%       72500.000000
50%      121200.000000
75%      179750.000000
max      633000.000000
Name: salary_midpoint, dtype: float64

Skill Counts:
Python jobs: 855
SQL jobs: 747

Remote / Hybrid Counts:
Remote jobs: 321
Hybrid jobs: 222

Experience vs Salary Correlation:
                  experience_years  salary_midpoint
experience_years             1.000            0.154
salary_midpoint              0.154            1.000

Employment Type Distribution:
employment_type
Full-time     1022
Contract        98
Part-time       38
Intern          14
Volunteer       12
TEMPORARY        8
OTHER            2
Internship       1
Name: count, dtype: Int64

Top Companies:
company
ADF Medical             31
Meta                    28
Cymertek Corporation    20
Mindrift                17
Netflix                 16
Haystack                12
DataAnnotation          10
Uber                    10


## EDA (Exploratory Data Analysis) — Notes

- Analyzed salary distribution (mean, median, range)
- Observed variation in salaries across roles
- Identified Python as most in-demand skill
- Identified SQL as second most in-demand skill
- Compared remote vs non-remote job availability
- Examined relationship between experience and salary
- Observed positive/weak correlation between experience and salary
- Analyzed job distribution by employment type
- Identified top companies with most job postings
- Identified top industries in dataset
- Compared average salary across industries and employment types